In [1]:
# MQA

import torch
import torch.nn as nn
import math


In [2]:
class MQA(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        
        self.d_model = d_model
        self.num_heads = num_heads
        
        assert d_model%num_heads==0, "d_model must divisible by num_heads"
        self.head_dim = d_model//num_heads
        
        self.qeury = nn.Linear(d_model, self.d_model)
        self.key = nn.Linear(d_model, self.head_dim) 
        self.val = nn.Linear(d_model, self.head_dim)
        
        self.linear = nn.Linear(d_model, d_model)
    
    def forward(self, x):
        batch_size, seq_len, d_model = x.size()
        query = self.qeury(x) 
        key = self.key(x).unsqueeze(1) # batch_size, 1,seq_len, head_dim
        val = self.val(x).unsqueeze(1) #batch_size, 1, seq_len, head_dim
        
        Q = query.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1,2)  # [batch_size, num_heads, seq_length, head_dim]
        print(Q.size())
        print(key.transpose(-2,-1).size())
        # attention
        scores = torch.matmul(Q, key.transpose(-2,-1))/ math.sqrt(d_model)
        print(scores.size())
        
        scores = torch.softmax(scores,dim=-1) # batch_size, seq_len, seq_len
        attn_output = torch.matmul(scores, val) #batch_size, num_heads, seq_len, head_dim
        
        attn_output = attn_output.transpose(1,2).contiguous().view(batch_size, seq_len, self.num_heads*self.head_dim)
        
        attn_output = self.linear(attn_output)
        
        return attn_output, scores

In [3]:
batch_size, seq_len, d_model = 16, 10, 768

mqa = MQA(d_model, 12)

x = torch.randn(batch_size, seq_len, d_model)

output, _ = mqa(x)

print(f"output is {output.size()}")

torch.Size([16, 12, 10, 64])
torch.Size([16, 1, 64, 10])
torch.Size([16, 12, 10, 10])
output is torch.Size([16, 10, 768])


In [4]:
class GQA(nn.Module):
    def __init__(self, d_model, head_dim, num_q_heads, num_kv_groups=None):
        super().__init__()
        self.d_model = d_model
        self.head_dim = head_dim
        self.num_kv_groups = num_kv_groups
        self.num_q_heads = num_q_heads
        
        assert num_q_heads%num_kv_groups==0, "num_q_heads must be divisible by num_kv_groups"
        self.query = nn.Linear(d_model, d_model)
        self.key = nn.Linear(d_model, num_kv_groups*head_dim)
        self.val = nn.Linear(d_model, num_kv_groups*head_dim)
        
        self.out_proj = nn.Linear(d_model, d_model)
        
    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        Q = self.query(x) # batch_szie, seq_len, d_model
        K = self.key(x)   # batch_size, seq_len, num_kv_groups*head_dim
        V = self.val(x)   # batch_szie, seq_len, num_kv_groups*head_dim

        # Q*KT
        # Q ：batch_size, num_q_heads, seq_len, head_dim
        # K: batch_size, num_kv_groups, seq_len, head_dim
        Q = Q.view(batch_size, seq_len, self.num_q_heads, head_dim).transpose(1,2)
        K = K.view(batch_size, seq_len, self.num_kv_groups, self.head_dim).transpose(1,2)
        K = torch.repeat_interleave(K, self.num_q_heads//self.num_kv_groups, 1) # batch_size, num_q_heads, seq_len, head_dim
        
        scores = torch.matmul(Q, K.transpose(-2,-1))/math.sqrt(head_dim)
        # batch_size, num_q_heads, seq_len, seq_len
        # V batch_szie, seq_len, num_kv_groups*head_dim
        
        V = V.view(batch_size, seq_len, num_kv_groups, head_dim).transpose(1,2)
        V = torch.repeat_interleave(V, self.num_q_heads//self.num_kv_groups, 1) #batch_size, num_q_heads, seq_len, head_dim
        scores = torch.softmax(scores, dim = -1)
        
        attn_out = torch.matmul(scores, V)
        attn_out = attn_out.transpose(1,2).contiguous().view(batch_size, seq_len, num_q_heads*head_dim)
        
        output = self.out_proj(attn_out)
        
        return output, scores

In [5]:
batch_size, num_kv_groups, seq_len, head_dim = 1,2,5,8
num_q_heads = 4
# batch_size, num_kv_groups, seq_len, head_dim ->
#batch_size, num_q_heads, seq_len, head_dim
x = torch.randn(batch_size, num_kv_groups, seq_len, head_dim)
x[0]

tensor([[[-0.0085, -0.9192, -1.2371, -0.5504, -0.0364,  0.3489,  0.3582,
           1.8566],
         [ 0.7580,  0.6997, -0.1227, -0.4615, -1.1965,  1.5074,  0.6319,
          -0.9588],
         [-0.3323, -0.8466,  1.0430,  1.6230,  0.6905,  0.9458, -0.2085,
          -0.1858],
         [ 1.3947,  1.5474,  1.9475,  0.8802, -0.5392,  0.8921, -0.0162,
          -1.6483],
         [ 0.4255, -1.3218, -0.4244,  0.6455,  1.4793,  2.6357,  0.7903,
           1.0724]],

        [[-0.4409,  2.6760, -0.7920,  0.0641,  0.1741,  0.0868, -0.3394,
          -1.4545],
         [ 0.0101, -0.1645, -0.5987,  0.1162, -1.8322, -0.0241, -0.2322,
           0.4972],
         [ 0.6819,  0.1043, -1.3754,  0.8846, -0.6756, -0.1447, -1.1558,
          -0.4744],
         [ 1.8672, -0.9013,  1.2515,  0.0804, -1.2894,  1.2183, -0.4008,
          -1.6045],
         [ 1.4729,  0.8487, -0.3116, -0.2435,  0.2225,  0.3504,  0.4292,
          -1.3997]]])

In [6]:
torch.repeat_interleave(x, 2, 1).size()

torch.Size([1, 4, 5, 8])

In [7]:
batch_size, seq_len, d_model = 16, 10, 768
gqa = GQA(d_model, 64 ,12, 4)

x = torch.randn(batch_size, seq_len, d_model)

output, _ = mqa(x)

print(f"output is {output.size()}")

torch.Size([16, 12, 10, 64])
torch.Size([16, 1, 64, 10])
torch.Size([16, 12, 10, 10])
output is torch.Size([16, 10, 768])


In [8]:
768//12

64